In [1]:
#r "/home/qinglei/Projects/Polars.NET/Polars.FSharp/bin/Debug/net8.0/Polars.FSharp.dll"
#r "/home/qinglei/Projects/Polars.NET/Polars.NET.Core/bin/Debug/net8.0/Polars.NET.Core.dll"
#r "nuget: Apache.Arrow, 23.0.0"
#r "nuget: Apache.Arrow.Adbc"

open Microsoft.DotNet.Interactive.Formatting
open System
open Polars.FSharp
let display (x: obj) = x
Display.init()

Installed Packages Apache.Arrow, 23.0.0 Apache.Arrow.Adbc, 0.23.0

In [2]:
// Define data
type WeatherData = { Date: string; City: string; Temperature: float; Rain: bool }

let data = [
    { Date="2023-01-01"; City="London";     Temperature=10.5; Rain=true }
    { Date="2023-01-01"; City="Manchester"; Temperature=9.0;  Rain=true }
    { Date="2023-01-02"; City="London";     Temperature=12.1; Rain=false }
]

let df = 
    DataFrame.ofRecords data
    |> pl.withColumn (pl.col("Date").Str.ToDate "%Y-%m-%d")

df

Datedate,Citystr,Temperaturef64,Rainbool
2023-01-01,"""London""",10.5,true
2023-01-01,"""Manchester""",9,true
2023-01-02,"""London""",12.1,false


In [3]:
let df_converted =
    df 
    |> pl.filter (pl.col "City" .== pl.lit "London")
    |> pl.select [
        pl.col("Date")
        (pl.col("Temperature") * pl.lit 1.8 + pl.lit 32.0) |> pl.alias "Temp_F"
    ]

df_converted

Datedate,Temp_Ff64
2023-01-01,50.9
2023-01-02,53.78


In [4]:
let londonDf = 
    df 
    |> pl.filter (pl.col "City" .== pl.lit "London")

londonDf

Datedate,Citystr,Temperaturef64,Rainbool
2023-01-01,"""London""",10.5,true
2023-01-02,"""London""",12.1,false


In [5]:
let aggDf = 
    df
    |> pl.groupBy [pl.col "City"]
    |> pl.agg [
        pl.col("Temperature").Mean().Alias "Avg_Temp"
        pl.col("Temperature").Max().Alias "Max_Temp"
        pl.col("Rain").Sum().Alias "Rainy_Days" // bool sum -> count of true
        pl.len().Alias "Total_Records"
    ]

aggDf

Citystr,Avg_Tempf64,Max_Tempf64,Rainy_Daysu32,Total_Recordsu32
"""Manchester""",9,9,1,1
"""London""",11.3,12.1,1,2


In [6]:
let windowDf = 
    df
    |> pl.select [
        pl.col "Date"
        pl.col "City"
        pl.col "Temperature"
        
        // Over(Date): Calculate mean value by group
        (pl.col("Temperature").Mean().Over(pl.col "Date"))
            |> pl.alias "Daily_Avg"
            
        (pl.col "Temperature" - pl.col("Temperature").Mean().Over(pl.col "Date"))
            |> pl.alias "Diff"
    ]
    |> pl.sortAscending [pl.col "Date"] 
windowDf


Datedate,Citystr,Temperaturef64,Daily_Avgf64,Difff64
2023-01-01,"""London""",10.5,9.75,0.75
2023-01-01,"""Manchester""",9,9.75,-0.75
2023-01-02,"""London""",12.1,12.1,0


In [7]:
let lf = 
    df 
    |> pl.asLazy
    |> pl.filterLazy (pl.col "Temperature" .> pl.lit 10.0)
    |> pl.groupByLazy [pl.col "City"] 
    |> pl.aggLazy [(pl.col "Temperature").Mean() |> pl.alias "Lazy_Avg_Temp"]   
    
printfn "%s" (lf.Explain(optimized=true))

let finalDf = lf.Collect()

finalDf

AGGREGATE[maintain_order: false]
  [col("Temperature").mean().alias("Lazy_Avg_Temp")] BY [col("City")]
  FROM
  FILTER [(col("Temperature")) > (10.0)]
  FROM
    DF ["Date", "City", "Temperature", "Rain"]; PROJECT["Temperature", "City"] 2/4 COLUMNS


Citystr,Lazy_Avg_Tempf64
"""London""",11.3


In [8]:
let data = [Some "10.50"; Some "20.25"; None]
let s = pl.series "str_vals" data

let sDec = s.Cast(DataType.Decimal(10,2))

let logic (opt: decimal option) =
    opt |> Option.map (fun d -> d * 2m)

let res = sDec.MapOption(logic, DataType.Decimal(10, 2))

res.Show()

shape: (3,)
Series: 'str_vals' [decimal[38,18]]
[
	21.000000000000000000
	40.500000000000000000
	null
]


In [9]:
let result =
    [
        pl.series "Id"    [1; 2; 3]
        pl.series "Value" ["A"; "B"; "C"]
    ] 
    |> pl.dataframe |> pl.asLazy
    |> Merge.initiate 
        (
            [
            pl.series "Id"    [2; 3; 4]
            pl.series "Value" ["B_new"; "C_new"; "D"]
            ]
            |> pl.dataframe |> pl.asLazy
        ) 
        ["Id"]
    |> Merge.whenMatchedUpdateSet (Set.build [
        Set.col "Value" (fun ctx -> ctx.SourceCol "Value")
    ])
    |> Merge.whenNotMatchedInsertAll
    |> Merge.execute
    |> pl.sortAscendingLazy [pl.col "Id"]
    |> pl.collect

result

Idi32,Valuestr
1,"""A"""
2,"""B_new"""
3,"""C_new"""
4,"""D"""
